In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings

In [2]:
df = pd.read_csv('../datasets/NFL_schedule.csv')

In [4]:
df.columns

Index(['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday',
       'gametime', 'away_team', 'away_score', 'home_team', 'home_score',
       'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis',
       'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest',
       'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds',
       'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game',
       'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id',
       'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee',
       'stadium_id', 'stadium'],
      dtype='str')

In [6]:
df['div_game'].unique()

array([0, 1])

In [7]:
df['winner (home)'] = (df['home_score'] > df['away_score']).astype(int)

In [8]:
pd.set_option('display.max_columns', None)
df.tail(1)

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium,winner (home)
2760,2025_22_SEA_NE,2025,SB,22,2026-02-08,Sunday,18:30,SEA,29,NE,13,Neutral,-16,42,0,2026020800,60176,NaN,202602080nwe,NaN,401772988,NaN,14,14,-238.0,195.0,-4.5,-115.0,-105.0,45.5,-115.0,-105.0,0,outdoors,grass,67.0,7.0,00-0034869,00-0039851,Sam Darnold,Drake Maye,Mike Macdonald,Mike Vrabel,Shawn Smith,SFO01,Levi's Stadium,0


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score
X_pred = df[['season', 'week', 'game_type', 'away_team', 'home_team', 'away_rest', 'home_rest', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'div_game']]
X_pred = pd.get_dummies(data=X_pred, columns=['game_type', 'away_team', 'home_team'], drop_first=True, dtype=int)
X_pred.tail(1)


,season,week,away_rest,home_rest,spread_line,away_spread_odds,home_spread_odds,total_line,div_game,game_type_DIV,game_type_REG,game_type_SB,game_type_WC,away_team_ATL,away_team_BAL,away_team_BUF,away_team_CAR,away_team_CHI,away_team_CIN,away_team_CLE,away_team_DAL,away_team_DEN,away_team_DET,away_team_GB,away_team_HOU,away_team_IND,away_team_JAX,away_team_KC,away_team_LA,away_team_LAC,away_team_LV,away_team_MIA,away_team_MIN,away_team_NE,away_team_NO,away_team_NYG,away_team_NYJ,away_team_OAK,away_team_PHI,away_team_PIT,away_team_SD,away_team_SEA,away_team_SF,away_team_TB,away_team_TEN,away_team_WAS,home_team_ATL,home_team_BAL,home_team_BUF,home_team_CAR,home_team_CHI,home_team_CIN,home_team_CLE,home_team_DAL,home_team_DEN,home_team_DET,home_team_GB,home_team_HOU,home_team_IND,home_team_JAX,home_team_KC,home_team_LA,home_team_LAC,home_team_LV,home_team_MIA,home_team_MIN,home_team_NE,home_team_NO,home_team_NYG,home_team_NYJ,home_team_OAK,home_team_PHI,home_team_PIT,home_team_SD,home_team_SEA,home_team_SF,home_team_TB,home_team_TEN,home_team_WAS
2760,2025,22,14,14,-4.5,-115.0,-105.0,45.5,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [10]:
y = df['winner (home)']

In [12]:
X_pred = X_pred.fillna(X_pred.median(numeric_only=True))

In [13]:
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

X_train, X_test, y_train, y_test = train_test_split(
    X_pred, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

model_scaled = LogisticRegression(max_iter=2000)
model_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = model_scaled.predict(X_test_scaled)
y_scaled_proba = model_scaled.predict_proba(X_test_scaled)[:, 1]


print(f"ROC_AUC: {roc_auc_score(y_test, y_pred_proba)}")
print(f"Accuracy without scaling:  {accuracy_score(y_test, y_pred)}")
print(f"Confusion Matrix without scaling: {confusion_matrix(y_test, y_pred)}")

print(f"ROC_AUC_scaled: {roc_auc_score(y_test, y_scaled_proba)}")
print(f"Accuracy with scaling:  {accuracy_score(y_test, y_pred_scaled)}")
print(f"Confusion Matrix with scaling: {confusion_matrix(y_test, y_pred_scaled)}")

ROC_AUC: 0.6973597359735972
Accuracy without scaling:  0.6455696202531646
Confusion Matrix without scaling: [[137 113]
 [ 83 220]]
ROC_AUC_scaled: 0.6911287128712871
Accuracy with scaling:  0.6419529837251357
Confusion Matrix with scaling: [[135 115]
 [ 83 220]]


In [14]:
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'penalty' :trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet', None]),
        'C' : trial.suggest_float('C', 1e-4, 1e3, log=True),  
    }
    
    if params['penalty'] == 'elasticnet':
        params['l1_ratio'] = trial.suggest_float('l1_ratio', 0.0, 1.0)
    
    model = LogisticRegression(**params, solver='saga', max_iter=2000, random_state=101)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    return score.mean()
    

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)
print(study.best_value)
print(study.best_params)

0.6693744164332399
{'penalty': 'l2', 'C': 0.0009512383181544084}


In [15]:
best_parms = study.best_params
final_model = LogisticRegression(**best_parms, solver='saga', max_iter=2000, random_state=101)
final_model.fit(X_train, y_train)
y_pred_final = final_model.predict(X_test)
y_pred_final_proba = final_model.predict_proba(X_test)[:, 1]
print(f"ROC_AUC: {roc_auc_score(y_test, y_pred_final_proba)}")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_final)}")
print(f"Confusion Matrix without scaling: {confusion_matrix(y_test, y_pred_final)}")

ROC_AUC: 0.720105610561056
Accuracy:  0.6636528028933092
Confusion Matrix without scaling: [[134 116]
 [ 70 233]]
